# Final sweep benchmark — Blinatumomab

Final model sweep: **fusion architectures** (`baseline`, `S1_moe`, `S2_poetemp`, `S3_mvtcae`, `S4_mmvaeplus`) crossed with the ref-notebook params (`spatial × n_latent{20,30} × n_layers × batch_mask`), trained by `sweep/train_final_sweep.py` and cached in `cache/final_sweep/`.

This notebook (CPU aggregation, mirrors `sweep_benchmark.ipynb`) evaluates each run and selects the model(s) that are **on par with both single-modality scVI specialists**:

1. **Anchors** — every benchmark draws the two scVI specialists: on *abundance*, `scvi_abundance`=ceiling / `scvi_spatial`=floor; on *spatial*, reversed.
2. **Per-modality vs scVI** — each model's abundance latent (`z_abn`) is scored against `scvi_abundance`, its spatial latent (`z_spt`) against `scvi_spatial` (Moran's I + PPC).
3. **Joint benchmarks** — UMAP / PPC / joint Moran's I with the same anchors.
4. **Selection** — models on the Pareto front meeting **both** specialist ceilings → `cache/final_sweep/selected.json`.

> Run `sweep/train_scvi_baselines.py` (anchors) and the `sweep/submit_final_sweep.sh` grid first.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

PIXELGEN_ROOT = '/home/projects/nyosef/zvise/PixelGen'        # parent -> import as PixelGen.*
PIXELGEN_PKG  = PIXELGEN_ROOT + '/PixelGen'                    # repo dir -> metrics.py's `from scvi_utils import`
for p in (PIXELGEN_ROOT, PIXELGEN_PKG):
    if p not in sys.path:
        sys.path.append(p)

from PixelGen.metrics import distr_autocorrelation_in_latent
from PixelGen.scvi_utils import pca_neighbors_umap

NEW_DATA = Path(PIXELGEN_PKG) / 'New_Data'
CACHE    = NEW_DATA / 'cache'
FINAL    = CACHE / 'final_sweep'
ADATA_PATH = CACHE / 'adata_cytovi_annotated_compat.h5ad'

ABUNDANCE_REP = 'arcsinh'                      # abundance features for autocorrelation
SPATIAL_REP   = 'spatial_asinh5_top500var'    # spatial features for autocorrelation
BIO_KEY   = 'cell_type_annot'
BATCH_KEY = 'cell_system'
PCA_KW    = {'n_comps': 15}
TOL = 0.02    # "on par" tolerance vs the specialist ceiling

sc.set_figure_params(figsize=(5, 3), frameon=False)

## Load data + scVI specialist anchors

The next cell builds `cache/final_sweep/scvi_baselines.npz` (the `z_scvi_abundance` / `z_scvi_spatial` specialists) on first run if it's missing — that step trains two scVI models, so **run it on a GPU kernel**. Once cached it just loads. These two specialists are the per-modality best/worst envelope used in every benchmark below. (Equivalent to running `sweep/train_scvi_baselines.py` standalone.)

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(adata.shape, '|', BIO_KEY, adata.obs[BIO_KEY].nunique(), '|', BATCH_KEY, adata.obs[BATCH_KEY].nunique())

In [ ]:
# scVI specialist anchors — build once on a GPU kernel if missing.
# Params match spatial_pca_benchmark.ipynb EXACTLY: n_latent=20, n_hidden=128, n_layers=2,
# dropout_rate=0.1, gene_likelihood='normal', batch_key=None (no batch correction),
# inputs shifted to >=0, max_epochs=200, batch_size=512, early_stopping.
import anndata as ad
import scvi
from PixelGen.utils import get_dense

BASELINES_NPZ = FINAL / 'scvi_baselines.npz'
if not BASELINES_NPZ.exists():
    print('scvi_baselines.npz not found — training the two scVI specialists (GPU)…')
    scvi.settings.seed = 0
    SCVI_KWARGS  = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1, gene_likelihood='normal')
    TRAIN_KWARGS = dict(max_epochs=200, batch_size=512, early_stopping=True)

    def _train_scvi_baseline(X, var_names, name):
        Xd = np.asarray(get_dense(X), dtype='float32')
        Xd = np.clip(Xd - Xd.min(axis=0, keepdims=True), 0.0, None)   # scVI needs non-negative input
        a = ad.AnnData(X=Xd, obs=adata.obs.copy())
        a.var_names = [str(v) for v in var_names]
        scvi.model.SCVI.setup_anndata(a, batch_key=None)              # matches spatial_pca_benchmark
        m = scvi.model.SCVI(a, **SCVI_KWARGS)
        print(f'>>> training vanilla scVI {name}: n_vars={a.n_vars}')
        m.train(**TRAIN_KWARGS)
        return m.get_latent_representation()

    z_abn = _train_scvi_baseline(adata.layers[ABUNDANCE_REP], adata.var_names, 'abundance')
    z_spt = _train_scvi_baseline(adata.obsm[SPATIAL_REP], adata.obsm[SPATIAL_REP].columns, 'spatial')
    FINAL.mkdir(parents=True, exist_ok=True)
    np.savez(BASELINES_NPZ, z_scvi_abundance=z_abn, z_scvi_spatial=z_spt)
    print('saved', BASELINES_NPZ)

bl = np.load(BASELINES_NPZ)
adata.obsm['z_scvi_abundance'] = bl['z_scvi_abundance']
adata.obsm['z_scvi_spatial']   = bl['z_scvi_spatial']
print('anchors:', {k: adata.obsm[k].shape for k in ['z_scvi_abundance', 'z_scvi_spatial']})

## Train the sweep — submit 80 LSF GPU jobs

The 80 configs are trained by `sweep/train_final_sweep.py`, one GPU job each (idempotent — re-running only fills gaps). The next cell prints the `bsub` commands; set `SUBMIT = True` to fire them. After the jobs finish (each writes `<run_id>/DONE`), run the aggregation cells below. (Equivalent to `bash sweep/submit_final_sweep.sh` from a shell.)

In [ ]:
# Submit the 80-run final sweep as LSF GPU jobs (same grid as sweep/submit_final_sweep.sh).
# Idempotent: skips configs whose <run_id>/DONE already exists.
# Prints the bsub commands by default; set SUBMIT = True to actually fire them.
import subprocess
from itertools import product

SWEEP = Path(PIXELGEN_PKG) / 'New_Data' / 'sweep'
LOGS  = SWEEP / 'logs_final'
LOGS.mkdir(parents=True, exist_ok=True)
FINAL.mkdir(parents=True, exist_ok=True)

ARCH_G    = ['baseline', 'S1_moe', 'S2_poetemp', 'S3_mvtcae', 'S4_mmvaeplus']
SPATIAL_G = ['top500', 'ct100']
NLAT_G    = [20, 30]              # n_latent=10 dropped: per-modality latent must be >= scVI(20)
NLAY_G    = [2, 3]
BMASK_G   = ['ff', 'ft']

SUBMIT = True                   # <-- flip to True to actually submit the jobs

n_sub = n_skip = 0
for a, sp, nl, L, bm in product(ARCH_G, SPATIAL_G, NLAT_G, NLAY_G, BMASK_G):
    rid = f'arch-{a}__sp-{sp}__nl{nl}__L{L}__bm-{bm}'
    if (FINAL / rid / 'DONE').exists():
        n_skip += 1
        continue
    cmd = ['bsub', '-J', f'fsw-{rid}', '-q', 'long-gpu',
           '-gpu', 'num=1:j_exclusive=yes:gmem=16G',
           '-R', 'rusage[mem=64G]', '-R', 'affinity[thread*8]',
           '-o', str(LOGS / f'{rid}.out'), '-e', str(LOGS / f'{rid}.err'),
           '--', 'bash', str(SWEEP / 'run_final_sweep.lsf'),
           '--arch', a, '--spatial', sp, '--n-latent', str(nl),
           '--n-layers', str(L), '--batch-mask', bm]
    if SUBMIT:
        subprocess.run(cmd, check=True)
    else:
        print(' '.join(cmd))
    n_sub += 1

print(f'\n{"submitted" if SUBMIT else "DRY-RUN — set SUBMIT=True to fire"}: {n_sub}   skipped(DONE): {n_skip}')

## Aggregate final-sweep runs

For each finished run load `latents.npz` and stash the three latents in `obsm`: joint (`z__{rid}`), abundance (`zabn__{rid}`), spatial (`zspt__{rid}`). `config.json` → `runs_df`.

In [ ]:
available, cfg_rows = [], []
for d in sorted(FINAL.glob('arch-*')):
    rid = d.name
    if not (d / 'DONE').exists() or not (d / 'latents.npz').exists():
        continue
    z = np.load(d / 'latents.npz')
    adata.obsm[f'z__{rid}']    = z['z_joint']
    adata.obsm[f'zabn__{rid}'] = z['z_abn']
    adata.obsm[f'zspt__{rid}'] = z['z_spt']
    cfg_rows.append(json.loads((d / 'config.json').read_text()))
    available.append(rid)

assert cfg_rows, (f'No finished runs under {FINAL}. Submit the sweep (cell above, SUBMIT=True), '
                  f'wait for jobs to write <run_id>/DONE, then re-run this cell.')
runs_df = pd.DataFrame(cfg_rows).set_index('run_id')
print(f'{len(available)} / 80 runs available')
runs_df[['arch', 'spatial', 'n_latent', 'n_layers', 'batch_mask']].sort_values(
    ['arch', 'spatial', 'n_latent']).head(20)

## Per-modality latent quality vs scVI specialists (instruction 2)

For every run we score its **abundance** latent (`z_abn`) by the Moran's I of the abundance features (`arcsinh`) under that latent's kNN graph, and its **spatial** latent (`z_spt`) by the Moran's I of the spatial features (`spatial_asinh5_top500var`). Both scVI specialists are scored on the same features, giving per-modality **ceiling** (matching specialist) and **floor** (off-specialist) anchors.

Two summaries per latent: the **mean** Moran's I, and an **AUC** = P(a feature autocorrelates more strongly under this latent than under the matching scVI specialist) — the Mann-Whitney effect size vs the ceiling (0.5 ties the specialist, >0.5 beats it). Two figures: the first overlays the full per-feature Moran's I distribution for **all** latents (one histogram per run, exactly like `fusion_benchmark`); the second keeps only the two scVI anchors and the runs that **match or beat** the ceiling (AUC≥0.5).

A good multimodal model keeps *both* latents near their respective ceilings (AUC≈0.5+) — no modality collapse.

In [ ]:
import seaborn as sns
from scipy.stats import mannwhitneyu

def morans_auc(ac_df, anchor):
    """Per-method AUC = P(a feature's Moran's I is higher under the method than under `anchor`),
    the Mann-Whitney effect size vs the matching scVI specialist. 0.5 = ties the specialist,
    >0.5 = beats it (distribution shifted to higher autocorrelation)."""
    ref = ac_df.loc[ac_df['latent'] == anchor, 'morans'].to_numpy()
    return pd.Series({
        name: mannwhitneyu(g['morans'].to_numpy(), ref, alternative='two-sided').statistic
              / (len(g) * len(ref))
        for name, g in ac_df.groupby('latent')})

# Abundance latent quality: arcsinh-feature Moran's I under each model's z_abn (+ both anchors)
abn_keys  = [f'zabn__{r}' for r in available] + ['z_scvi_abundance', 'z_scvi_spatial']
abn_names = list(available)                  + ['scvi_abundance',   'scvi_spatial']
abn_ac    = distr_autocorrelation_in_latent(adata, abn_keys, abn_names,
                                            rep_key=ABUNDANCE_REP, pca_kwargs=PCA_KW)
abn_mean  = abn_ac.groupby('latent')['morans'].mean()
abn_auc   = morans_auc(abn_ac, 'scvi_abundance')      # vs the matching specialist (ceiling)

# Spatial latent quality: spatial-feature Moran's I under each model's z_spt (+ both anchors)
spt_keys  = [f'zspt__{r}' for r in available] + ['z_scvi_spatial',   'z_scvi_abundance']
spt_names = list(available)                  + ['scvi_spatial',     'scvi_abundance']
spt_ac    = distr_autocorrelation_in_latent(adata, spt_keys, spt_names,
                                            rep_key=SPATIAL_REP, pca_kwargs=PCA_KW)
spt_mean  = spt_ac.groupby('latent')['morans'].mean()
spt_auc   = morans_auc(spt_ac, 'scvi_spatial')        # vs the matching specialist (ceiling)

# Per-modality ceiling (matching specialist) / floor (off-specialist) anchors
ANCH = dict(
    abn_ceiling=abn_mean['scvi_abundance'], abn_floor=abn_mean['scvi_spatial'],
    spt_ceiling=spt_mean['scvi_spatial'],   spt_floor=spt_mean['scvi_abundance'],
)
# score columns: mean Moran's I + AUC vs matching specialist; config spatial-selection -> 'spatial_sel'
cfg = runs_df[['arch', 'spatial', 'n_latent', 'n_layers', 'batch_mask']].rename(
    columns={'spatial': 'spatial_sel'})
per_mod = pd.DataFrame({
    'abundance':     abn_mean.reindex(available),
    'abundance_auc': abn_auc.reindex(available),
    'spatial':       spt_mean.reindex(available),
    'spatial_auc':   spt_auc.reindex(available),
}).join(cfg)
print({k: round(v, 4) for k, v in ANCH.items()})
per_mod.sort_values('spatial_auc', ascending=False).head(12)

In [ ]:
# All runs in grey (one KDE per run); only the two scVI anchors drawn in color
arch_pal = dict(zip(sorted(runs_df['arch'].unique()),
                    plt.cm.tab10(np.linspace(0, 1, runs_df['arch'].nunique()))))  # used by later cells
anchor_col = {'scvi_abundance': 'tab:blue', 'scvi_spatial': 'tab:red'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (mod, ac) in zip(axes, [('abundance', abn_ac), ('spatial', spt_ac)]):
    for r in available:                                       # every model run: grey
        sns.kdeplot(x=ac.loc[ac['latent'] == r, 'morans'].to_numpy(), ax=ax,
                    color='0.6', lw=0.6, alpha=0.3)
    for a, c in anchor_col.items():                           # scVI anchors: colored + labeled
        sns.kdeplot(x=ac.loc[ac['latent'] == a, 'morans'].to_numpy(), ax=ax,
                    color=c, lw=2.5, label=a)
    ax.set(title=f"{mod} Moran's I — {len(available)} runs (grey) + scVI anchors", xlabel="Moran's I")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Simplified view: the two scVI anchors + only the runs that match/beat the matching specialist (AUC >= 0.5)
anchors = ['scvi_abundance', 'scvi_spatial']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (mod, ac, auc, ceil) in zip(axes, [
        ('abundance', abn_ac, abn_auc, 'scvi_abundance'),
        ('spatial',   spt_ac, spt_auc, 'scvi_spatial')]):
    wins = auc.reindex(available).dropna().loc[lambda s: s >= 0.5].sort_values(ascending=False)
    keep = anchors + list(wins.index)
    sns.histplot(data=ac[ac['latent'].isin(keep)], x='morans', hue='latent', hue_order=keep,
                 kde=True, stat='density', common_norm=False, alpha=0.35, ax=ax)
    ax.axvline(ac.loc[ac['latent'] == ceil, 'morans'].mean(), color='k', ls='--', lw=1)
    ax.set(title=f"{mod}: {len(wins)} runs >= scVI ceiling (AUC>=0.5)", xlabel="Moran's I")
    if ax.get_legend():
        sns.move_legend(ax, 'upper left', fontsize=6, title=None)
plt.tight_layout(); plt.show()

## Top-5 spatial runs among the abundance winners

Of the runs that match/beat the abundance ceiling (`abundance_auc ≥ 0.5`, the abundance-good models coloured above), the 5 with the highest spatial Moran's I. Their **spatial** per-feature Moran's I distributions are drawn against the scVI **spatial** specialist (ceiling) and the **abundance** specialist (floor).

In [ ]:
# Figure 3: top-5 spatial runs among the abundance winners — spatial Moran's I vs the scVI anchors
short = lambda r: r.replace('arch-', '').replace('__', '·')
pool = per_mod[per_mod['abundance_auc'] >= 0.5]                       # abundance winners (coloured earlier)
top5 = pool.sort_values('spatial', ascending=False).head(5).index.tolist()
print('top-5 spatial among abundance winners:')
print(per_mod.loc[top5, ['abundance_auc', 'spatial', 'spatial_auc']].round(3))

fig, ax = plt.subplots(figsize=(8, 5))
ref = spt_ac.loc[spt_ac['latent'] == 'scvi_spatial', 'morans'].to_numpy()
sns.histplot(x=ref, ax=ax, color='k', stat='density', bins=25, kde=True, fill=True,
             alpha=0.15, label='scvi_spatial (ceiling)')
ax.axvline(ref.mean(), color='k', ls='--', lw=1)
sns.kdeplot(x=spt_ac.loc[spt_ac['latent'] == 'scvi_abundance', 'morans'].to_numpy(), ax=ax,
            color='grey', ls=':', lw=1.5, label='scvi_abundance (floor)')
for r, c in zip(top5, plt.cm.tab10(np.linspace(0, 1, len(top5)))):
    sns.kdeplot(x=spt_ac.loc[spt_ac['latent'] == r, 'morans'].to_numpy(), ax=ax,
                color=c, lw=2, label=short(r))
ax.set(title="Top-5 spatial runs (abundance winners): spatial Moran's I vs scVI", xlabel="Moran's I")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

## scib metrics for the selected models (instruction 5)

Apply scib integration metrics to the 5 selected models against the **vanilla MM** baseline (strongest `arch-baseline` run) and the two scVI specialists. Each model contributes its **joint**, **abundance** and **spatial** latent; the scVI specialists are per-modality. Run **twice**: bio-preservation of `cell_type_annot` (batch = `cell_system`), and of `cell_system` (batch = `cell_type_annot`).

In [ ]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

vanilla = per_mod.query('arch == "baseline"')['spatial'].idxmax()    # strongest baseline = "vanilla MM"
scib_runs = list(top5) + [vanilla]

emb = {}                                                             # readable name -> obsm key
for r in scib_runs:
    tag = 'vanilla' if r == vanilla else short(r)
    emb[f'{tag}·joint'] = f'z__{r}'                                  # joint per model
    emb[f'{tag}·abn']   = f'zabn__{r}'
    emb[f'{tag}·spt']   = f'zspt__{r}'
emb['scvi_abundance'] = 'z_scvi_abundance'                           # per-modality specialists
emb['scvi_spatial']   = 'z_scvi_spatial'
for name, key in emb.items():                                        # readable obsm copies -> clean table/figure labels
    adata.obsm[name] = adata.obsm[key]

def run_scib(label_key, batch_key):
    bm = Benchmarker(adata, batch_key=batch_key, label_key=label_key,
                     embedding_obsm_keys=list(emb.keys()),
                     bio_conservation_metrics=BioConservation(),
                     batch_correction_metrics=BatchCorrection())
    bm.benchmark()
    return bm

print('vanilla MM =', vanilla, '|', len(emb), 'embeddings')
bm_celltype = run_scib(BIO_KEY,   BATCH_KEY)                          # bio = cell type,  batch = system
bm_system   = run_scib(BATCH_KEY, BIO_KEY)                            # bio = system,     batch = cell type
bm_celltype.get_results(min_max_scale=False).to_csv(FINAL / 'scib_top5_celltype.csv')
bm_system.get_results(min_max_scale=False).to_csv(FINAL / 'scib_top5_system.csv')
print("bio-preservation of cell_type_annot (batch = cell_system)")
bm_celltype.plot_results_table(min_max_scale=False)

In [ ]:
print("bio-preservation of cell_system (batch = cell_type_annot)")
bm_system.plot_results_table(min_max_scale=False)

In [ ]:
list(bm_celltype.get_results(min_max_scale=False).index)   # row names of the scib table

## Per-modality reconstruction (PPC)

Cached `ppc_metrics.parquet` (feature-level Pearson / MAE per modality, from `calculate_metrics`) per run. This is the self-reconstruction view of the same per-modality split; the scVI specialists reconstruct shifted inputs so they are not directly comparable here — the modality benchmark that carries the scVI anchors is the Moran's I above.

In [ ]:
ppc = pd.concat([pd.read_parquet(FINAL / r / 'ppc_metrics.parquet') for r in available],
                ignore_index=True)
ppc_feat = ppc[ppc['Level'] == 'Features']
ppc_pearson = ppc_feat.pivot_table(index='Model', columns='Modality', values='Pearson')
ppc_pearson = ppc_pearson.join(runs_df['arch']).sort_values('Spatial', ascending=False)
print('feature-level reconstruction Pearson (per modality)')
ppc_pearson.head(12)

## Joint-latent benchmarks (instruction 1)

Joint latent (`z__{rid}`) viewed with the two scVI anchors: UMAP organization (cell type vs system), and joint Moran's I on both feature sets.

In [ ]:
# UMAPs: the two scVI anchors + the best-spatial run of each architecture, colored by cell type & system
best_per_arch = (per_mod.sort_values('spatial', ascending=False)
                        .groupby('arch').head(1).index.tolist())
show = [('z_scvi_abundance', 'scvi_abundance'), ('z_scvi_spatial', 'scvi_spatial')] \
       + [(f'z__{r}', r) for r in best_per_arch]
for key, name in show:
    pca_neighbors_umap(adata, key,
                       umap_pl_kwargs=dict(color=[BIO_KEY, BATCH_KEY], ncols=2, show=False),
                       umap_title=name)
    plt.show()

In [ ]:
# Joint-latent Moran's I on both feature sets, with the scVI anchors
joint_keys  = [f'z__{r}' for r in available] + ['z_scvi_abundance', 'z_scvi_spatial']
joint_names = list(available)                + ['scvi_abundance',   'scvi_spatial']
jabn = distr_autocorrelation_in_latent(adata, joint_keys, joint_names,
                                       rep_key=ABUNDANCE_REP, pca_kwargs=PCA_KW
                                      ).groupby('latent')['morans'].mean()
jspt = distr_autocorrelation_in_latent(adata, joint_keys, joint_names,
                                       rep_key=SPATIAL_REP, pca_kwargs=PCA_KW
                                      ).groupby('latent')['morans'].mean()
joint_tbl = pd.DataFrame({'joint_abundance': jabn.reindex(available),
                          'joint_spatial':   jspt.reindex(available)}).join(runs_df['arch'])
print('anchors  abundance:', round(jabn['scvi_abundance'], 4),
      '| spatial:', round(jspt['scvi_spatial'], 4))
joint_tbl.sort_values('joint_spatial', ascending=False).head(12)

## Selection: on par with both specialists (instruction 4)

A run is **selected** if its abundance latent reaches the `scvi_abundance` ceiling *and* its spatial latent reaches the `scvi_spatial` ceiling (within `TOL`). The scatter places every run against the two ceilings; the target region is the top-right quadrant. Selected run_ids → `cache/final_sweep/selected.json`.

In [ ]:
abn_ceil, spt_ceil = ANCH['abn_ceiling'], ANCH['spt_ceiling']
sel = per_mod.copy()
sel['abn_ratio'] = sel['abundance'] / abn_ceil
sel['spt_ratio'] = sel['spatial']   / spt_ceil
sel['on_par'] = (sel['abundance'] >= abn_ceil * (1 - TOL)) & (sel['spatial'] >= spt_ceil * (1 - TOL))
sel['combined'] = sel[['abn_ratio', 'spt_ratio']].min(axis=1)   # weakest modality drives the score
selected = sel[sel['on_par']].sort_values('combined', ascending=False)
if selected.empty:                                              # fallback: closest to meeting both
    selected = sel.sort_values('combined', ascending=False).head(3)
    print('NOTE: no run met both ceilings within TOL — falling back to top-3 by weakest-modality ratio')

fig, ax = plt.subplots(figsize=(6, 5))
for a, g in sel.groupby('arch'):
    ax.scatter(g['abundance'], g['spatial'], s=28, color=arch_pal[a], label=a)
ax.scatter(sel.loc[selected.index, 'abundance'], sel.loc[selected.index, 'spatial'],
           s=120, facecolors='none', edgecolors='red', linewidths=1.5, label='selected')
ax.axvline(abn_ceil, ls='--', c='k', lw=1); ax.axhline(spt_ceil, ls='--', c='k', lw=1)
ax.scatter([ANCH['abn_floor']], [spt_ceil], marker='*', s=200, c='k')         # scvi_spatial
ax.scatter([abn_ceil], [ANCH['spt_floor']], marker='*', s=200, c='dimgrey')   # scvi_abundance
ax.set(xlabel="abundance Moran's I  (dashed = scvi_abundance ceiling)",
       ylabel="spatial Moran's I  (dashed = scvi_spatial ceiling)",
       title='On-par region = top-right quadrant (both ceilings met)')
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

(FINAL / 'selected.json').write_text(json.dumps({
    'selected': selected.index.tolist(),
    'ceilings': {'abundance': float(abn_ceil), 'spatial': float(spt_ceil)},
    'tol': TOL,
    'scores': selected[['abundance', 'spatial', 'abn_ratio', 'spt_ratio', 'combined']].round(4)
                      .to_dict('index'),
}, indent=2))
print(f'wrote {FINAL / "selected.json"}  ({len(selected)} selected)')

In [ ]:
# Selected models + their configs and per-modality scores
selected.join(joint_tbl[['joint_abundance', 'joint_spatial']])[
    ['arch', 'spatial_sel', 'n_latent', 'n_layers', 'batch_mask',
     'abundance', 'spatial', 'joint_abundance', 'joint_spatial', 'combined']]